## New Jersey Housing Market Crisis Analysis - Final by Felipe Viana


## Imports
- `requests` is used to call APIs.
- `pandas` is used to clean and analyze the data.
- `matplotlib` is used for visualizations.
- `json` is used to export the final cleaned project data.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import json
import os
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

For the project show the api's and my issues
Focus on the data collection - focus more on the sources
Show api websites and scema
Analysis show 1 or 2
Re focus.
Rent vs mortgage comparison - 

## API Keys
This project uses the Census API and FRED API.
- Census API key: https://api.census.gov/data/key_signup.html
- FRED API key: https://fredaccount.stlouisfed.org/apikey

In [ ]:
## API Keys
CENSUS_API_KEY = "insert your Census API key here"
FRED_API_KEY = "insert your FRED API key here"

## Helper Function for API Requests

In [ ]:
def get_json_response(url, params):
    response = requests.get(url, params=params)
    print("URL:", response.url)
    print("Status code:", response.status_code)
    print("Preview:", response.text[:200])
    try:
        return response.json()
    except Exception as error:
        print("JSON conversion failed.")
        print("Reason:", error)
        print("The API probably returned HTML or an error message instead of JSON.")
        return None

## County-Level Housing Affordability Data
This section collects county-level housing and income data from the U.S. Census ACS API.
The variables used include:
- `B19013_001E` = Median household income
- `B25077_001E` = Median home value
- `B25064_001E` = Median gross rent
- `B01003_001E` = Total population
- `B17001_002E` = People below poverty level
These variables help compare housing costs against local income and economic conditions.


In [ ]:
census_url = "https://api.census.gov/data/2023/acs/acs5"
census_params = {
    "get": "NAME,B19013_001E,B25077_001E,B25064_001E,B01003_001E,B17001_002E",
    "for": "county:*",
    "in": "state:34"
}
if CENSUS_API_KEY != "PASTE_YOUR_CENSUS_KEY_HERE" and CENSUS_API_KEY.strip() != "":
    census_params["key"] = CENSUS_API_KEY
census_data = get_json_response(census_url, census_params)
census_df = pd.DataFrame(census_data[1:], columns=census_data[0])
census_df.head()

## Cleaning Census Data

In [ ]:
census_df.columns = [
    "county_name",
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "population",
    "people_in_poverty",
    "state",
    "county_code"
]
numeric_cols = [
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "population",
    "people_in_poverty"
]
for col in numeric_cols:
    census_df[col] = pd.to_numeric(census_df[col], errors="coerce")
census_df = census_df.dropna()
census_df.head()

## Creating Housing Affordability Metrics

In [ ]:
census_df["monthly_income"] = census_df["median_household_income"] / 12
census_df["home_price_to_income_ratio"] = (
    census_df["median_home_value"] / census_df["median_household_income"]
)
census_df["rent_to_income_ratio"] = (
    census_df["median_gross_rent"] / census_df["monthly_income"]
)
census_df["poverty_rate"] = (
    census_df["people_in_poverty"] / census_df["population"]
) * 100
def affordability_label(ratio):
    if ratio < 3:
        return "Affordable"
    elif ratio < 5:
        return "Moderate"
    elif ratio < 7:
        return "Unaffordable"
    else:
        return "Severely Unaffordable"
census_df["affordability_status"] = census_df["home_price_to_income_ratio"].apply(affordability_label)
census_df.head()

## Least Affordable Counties

In [ ]:
worst_counties = census_df.sort_values(
    by="home_price_to_income_ratio",
    ascending=False
)
worst_counties[[
    "county_name",
    "median_household_income",
    "median_home_value",
    "home_price_to_income_ratio",
    "affordability_status"
]].head(10)

## Chart: Least Affordable Counties

In [ ]:
top10_affordable = worst_counties.head(10)
plt.figure(figsize=(12,6))
plt.barh(
    top10_affordable["county_name"],
    top10_affordable["home_price_to_income_ratio"]
)
plt.xlabel("Home Price to Income Ratio")
plt.ylabel("County")
plt.title("Least Affordable Counties in New Jersey")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Chart: Highest Median Home Values

In [ ]:
top_home_values = census_df.sort_values(
    by="median_home_value",
    ascending=False
).head(10)
plt.figure(figsize=(12,6))
plt.barh(
    top_home_values["county_name"],
    top_home_values["median_home_value"]
)
plt.title("Highest Median Home Values in New Jersey")
plt.xlabel("Median Home Value ($)")
plt.ylabel("County")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Rent by County

In [ ]:
rent_burden = census_df.sort_values(
    by="rent_to_income_ratio",
    ascending=False
).head(10)
plt.figure(figsize=(12,6))
plt.barh(
    rent_burden["county_name"],
    rent_burden["rent_to_income_ratio"]
)
plt.title("Counties with Highest Rent Burden")
plt.xlabel("Rent to Income Ratio")
plt.ylabel("County")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#percentage of rent vs mortgage + Hoa
#Stat bar graph


## Chart: Income vs Home Value

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(
    census_df["median_household_income"],
    census_df["median_home_value"]
)
plt.title("Income vs Home Value by County")
plt.xlabel("Median Household Income ($)")
plt.ylabel("Median Home Value ($)")
plt.grid(True)
plt.tight_layout()
plt.show()

## Chart: Poverty Rate vs Housing Affordability

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(
    census_df["poverty_rate"],
    census_df["home_price_to_income_ratio"]
)
plt.title("Poverty Rate vs Housing Affordability")
plt.xlabel("Poverty Rate (%)")
plt.ylabel("Home Price to Income Ratio")
plt.grid(True)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
census_df[[
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "home_price_to_income_ratio",
    "rent_to_income_ratio",
    "poverty_rate"
]].describe()

## Correlation Matrix

In [ ]:
census_df[[
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "home_price_to_income_ratio",
    "rent_to_income_ratio",
    "poverty_rate"
]].corr()

## Historical Housing, Jobs, Wages, Mortgage Rates, and Inflation

The series used from FRED API are:
- `NJSTHPI` = New Jersey House Price Index
- `NJUR` = New Jersey Unemployment Rate
- `SMU34000000500000003` = New Jersey Average Hourly Earnings
- `MORTGAGE30US` = 30-Year Fixed Mortgage Rate
- `CPIAUCSL` = Consumer Price Index


## FRED Data

In [ ]:
def get_fred_series(series_id, value_name):
    fred_url = "https://api.stlouisfed.org/fred/series/observations"
    fred_params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json"
    }
    data = get_json_response(fred_url, fred_params)
    if data is None or "observations" not in data:
        print(f"Could not load series: {series_id}")
        return pd.DataFrame()
    df = pd.DataFrame(data["observations"])
    df = df[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df[value_name] = pd.to_numeric(df["value"], errors="coerce")
    df = df.drop(columns=["value"])
    df = df.dropna()
    return df

## Collecting FRED Economic Data

In [ ]:
housing_df = get_fred_series("NJSTHPI", "house_price_index")
unemployment_df = get_fred_series("NJUR", "unemployment_rate")
wages_df = get_fred_series("SMU34000000500000003", "average_hourly_earnings")
mortgage_df = get_fred_series("MORTGAGE30US", "mortgage_rate")
cpi_df = get_fred_series("CPIAUCSL", "cpi")
print("Housing rows:", len(housing_df))
print("Unemployment rows:", len(unemployment_df))
print("Wage rows:", len(wages_df))
print("Mortgage rows:", len(mortgage_df))
print("CPI rows:", len(cpi_df))
housing_df.head()

## Calculating Percent Growth

In [ ]:
def add_percent_growth(df, value_col, new_col):
    df = df.copy()

    print("Available columns:", df.columns.tolist())

    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[value_col])

    df[new_col] = (
        (df[value_col] - df[value_col].iloc[0]) / df[value_col].iloc[0]
    ) * 100

    df["year"] = df["date"].dt.year

    yearly_df = df.groupby("year")[new_col].mean().reset_index()

    return df, yearly_df

## Overall Housing Price Growth

In [ ]:
start_price = housing_df["house_price_index"].iloc[0]
end_price = housing_df["house_price_index"].iloc[-1]
housing_percent_increase = (
    (end_price - start_price) / start_price
) * 100
print("Starting HPI:", round(start_price, 2))
print("Latest HPI:", round(end_price, 2))
print("Total Housing Price Increase:", round(housing_percent_increase, 2), "%")

## Chart: New Jersey Housing Price Growth Over Time

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    yearly_housing["year"],
    yearly_housing["housing_percent_growth"],
    linewidth=3
)
plt.title("Growth of New Jersey Housing Prices Over Time")
plt.xlabel("Year")
plt.ylabel("Percent Increase Since Earliest Year")
plt.xticks(
    range(
        yearly_housing["year"].min(),
        yearly_housing["year"].max() + 1,
        5
    ),
    rotation=45
)
plt.grid(True)
plt.tight_layout()
plt.show()


#see if covid had  an impact (covid intervention analysis) slopes comparison of pre and post covid growth rates

## Chart: New Jersey Unemployment Rate Over Time

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    unemployment_df["date"],
    unemployment_df["unemployment_rate"],
    linewidth=2
)
plt.title("New Jersey Unemployment Rate Over Time")
plt.xlabel("Year")
plt.ylabel("Unemployment Rate (%)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
print(wages_df.columns.tolist())
print(wages_df)
print(type(wages_df))

## Chart: Housing Growth vs Wage Growth(still trying to fix it)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    yearly_housing["year"],
    yearly_housing["housing_percent_growth"],
    linewidth=3,
    label="Housing Prices"
)
plt.plot(
    yearly_wages["year"],
    yearly_wages["wage_percent_growth"],
    linewidth=3,
    label="Wages"
)
plt.title("Housing Price Growth vs Wage Growth in New Jersey")
plt.xlabel("Year")
plt.ylabel("Percent Increase Since Earliest Year")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
print(yearly_housing.head())
print(yearly_wages.head())
print(wages_df.columns)


## Chart: 30-Year Mortgage Rates Over Time

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    mortgage_df["date"],
    mortgage_df["mortgage_rate"],
    linewidth=2
)
plt.title("30-Year Fixed Mortgage Rates Over Time")
plt.xlabel("Year")
plt.ylabel("Mortgage Rate (%)")
plt.grid(True)
plt.tight_layout()
plt.show()

## Chart: Housing Growth vs Inflation (Need to fix)

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(
    yearly_housing["year"],
    yearly_housing["housing_percent_growth"],
    linewidth=3,
    label="Housing Prices"
)
plt.plot(
    yearly_cpi["year"],
    yearly_cpi["inflation_percent_growth"],
    linewidth=3,
    label="Inflation"
)
plt.title("Housing Price Growth vs Inflation")
plt.xlabel("Year")
plt.ylabel("Percent Increase Since Earliest Year")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Final Project Data Export to JSON

In [ ]:
project_data = {
    "census_county_housing_data": census_df.to_dict(orient="records"),
    "housing_price_index_data": housing_df.to_dict(orient="records"),
    "unemployment_data": unemployment_df.to_dict(orient="records"),
    "wage_data": wages_df.to_dict(orient="records"),
    "mortgage_rate_data": mortgage_df.to_dict(orient="records"),
    "inflation_data": cpi_df.to_dict(orient="records")
}

with open("nj_housing_market_project_data.json", "w") as file:
    json.dump(project_data, file, indent=4, default=str)
print("JSON file saved successfully.")
print("Saved file:", os.path.abspath("nj_housing_market_project_data.json"))

In [ ]:
#Get the heat map working

import seaborn as sns
import matplotlib.pyplot as plt

correlation = census_df[[
    "median_household_income",
    "median_home_value",
    "median_gross_rent",
    "home_price_to_income_ratio"
]].corr()

plt.figure(figsize=(10,8))

sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=1,
    square=True,
    cbar_kws={"label": "Correlation"}
)

plt.title(
    "Correlation Between Housing Variables",
    fontsize=16,
    pad=20
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()